DATASET GENERATION

In [137]:
import numpy as np
import pandas as pd

def generate_synthetic_triage_data(n_samples=6230, random_seed=42):
    np.random.seed(random_seed)

    patient_ids = [f"PID_{i+1:05d}" for i in range(n_samples)]

    # 1. Demographics & Age Segmentation
    # ~15% pediatric (<18), ~55% adult (18-64), ~30% geriatric (65+)
    age_cohort = np.random.choice(['pediatric', 'adult', 'geriatric'], size=n_samples, p=[0.15, 0.55, 0.30])
    ages = np.empty(n_samples, dtype=int)
    ages[age_cohort == 'pediatric'] = np.random.randint(1, 18, size=np.sum(age_cohort == 'pediatric'))
    ages[age_cohort == 'adult'] = np.random.randint(18, 65, size=np.sum(age_cohort == 'adult'))
    ages[age_cohort == 'geriatric'] = np.random.randint(65, 95, size=np.sum(age_cohort == 'geriatric'))

    genders = np.random.choice(['Female', 'Male'], size=n_samples, p=[0.47, 0.53])

    # Clinical Frailty Scale (CFS: 1 to 9) - baseline median ~2 for younger, higher for geriatrics
    cfs_scores = np.ones(n_samples, dtype=int)
    cfs_scores[age_cohort == 'adult'] = np.random.choice([1, 2, 3, 4], size=np.sum(age_cohort == 'adult'), p=[0.5, 0.3, 0.15, 0.05])
    cfs_scores[age_cohort == 'geriatric'] = np.random.choice(range(1, 10), size=np.sum(age_cohort == 'geriatric'),
                                                            p=[0.05, 0.15, 0.25, 0.20, 0.15, 0.10, 0.05, 0.03, 0.02])

    # 2. Intake History Availability (50% zero-history / unverified per brief)
    has_prior_history = np.random.choice([1, 0], size=n_samples, p=[0.50, 0.50])
    comorbidity_counts = np.where(has_prior_history == 1, np.random.poisson(lam=np.where(age_cohort == 'geriatric', 2.5, 0.8)), 0)

    # 3. Base Clinical Presentations & ESI-v4 Baseline Ground Truth
    # ESI-v4 Distribution matching baseline: ESI 1 (~2.3%), ESI 2 (~27.1%), ESI 3 (~41.7%), ESI 4 (~27.0%), ESI 5 (~1.9%)
    esi_v4 = np.random.choice([1, 2, 3, 4, 5], size=n_samples, p=[0.023, 0.271, 0.417, 0.270, 0.019])

    # 4. Generate Age-Calibrated Physiological Vitals based on true acuity
    heart_rate = np.empty(n_samples)
    resp_rate = np.empty(n_samples)
    spo2 = np.empty(n_samples)
    sbp = np.empty(n_samples)
    temp_c = np.empty(n_samples)

    for i in range(n_samples):
        acuity = esi_v4[i]
        cohort = age_cohort[i]
        age = ages[i]

        # Base vital parameters by cohort
        if cohort == 'pediatric':
            base_hr = 110 if age < 5 else 90
            base_rr = 28 if age < 5 else 20
            base_sbp = 90 + (2 * age)
        elif cohort == 'geriatric':
            base_hr = 72
            base_rr = 18
            base_sbp = 135
        else: # adult
            base_hr = 75
            base_rr = 16
            base_sbp = 120

        # Vitals distortion based on ESI level
        if acuity == 1: # Resuscitation / Unstable
            hr = np.random.choice([np.random.normal(145, 15), np.random.normal(40, 8)])
            rr = np.random.choice([np.random.normal(36, 6), np.random.normal(6, 2)])
            sat = np.random.uniform(70, 88)
            bp = np.random.normal(base_sbp - 40, 15)
            temp = np.random.normal(38.8, 1.0)
        elif acuity == 2: # High Risk / Emergent
            hr = np.random.normal(base_hr + 35, 12)
            rr = np.random.normal(base_rr + 8, 4)
            sat = np.random.uniform(88, 94)
            bp = np.random.normal(base_sbp + 15, 20)
            temp = np.random.normal(38.2, 0.8)
        elif acuity == 3: # Urgent (Multiple resources, borderline vitals)
            # A subset will have abnormal vitals triggering ESI-v5 uptriage (~18% of level 3s)
            has_abnormal_vital = np.random.rand() < 0.18
            hr = np.random.normal(108, 8) if has_abnormal_vital else np.random.normal(base_hr + 10, 10)
            rr = np.random.normal(24, 3) if has_abnormal_vital else np.random.normal(base_rr + 2, 3)
            sat = np.random.uniform(90, 93) if has_abnormal_vital else np.random.uniform(95, 99)
            bp = np.random.normal(base_sbp, 15)
            temp = np.random.normal(37.4, 0.6)
        elif acuity == 4: # Less Urgent (1 resource)
            # ~10% have mild high-risk vital signs
            has_abnormal_vital = np.random.rand() < 0.10
            hr = np.random.normal(104, 5) if has_abnormal_vital else np.random.normal(base_hr, 8)
            rr = np.random.normal(22, 2) if has_abnormal_vital else np.random.normal(base_rr, 2)
            sat = np.random.uniform(91, 93) if has_abnormal_vital else np.random.uniform(96, 100)
            bp = np.random.normal(base_sbp, 10)
            temp = np.random.normal(36.8, 0.4)
        else: # ESI 5 (Non-urgent, 0 resources)
            hr = np.random.normal(base_hr, 6)
            rr = np.random.normal(base_rr, 2)
            sat = np.random.uniform(97, 100)
            bp = np.random.normal(base_sbp, 8)
            temp = np.random.normal(36.6, 0.3)

        heart_rate[i] = max(30, min(220, hr))
        resp_rate[i] = max(4, min(60, rr))
        spo2[i] = max(60.0, min(100.0, sat))
        sbp[i] = max(50, min(240, bp))
        temp_c[i] = max(34.0, min(41.5, temp))

    # 5. Deterministic ABCDE Floor & High-Risk Vital Flags
    # Adult limits: HR > 100, RR > 20, SpO2 < 92%. Pediatric limits adjusted by age.
    hr_flag = np.where(age_cohort == 'pediatric', (ages < 5) & (heart_rate > 140) | (ages >= 5) & (heart_rate > 110), heart_rate > 100)
    rr_flag = np.where(age_cohort == 'pediatric', (ages < 5) & (resp_rate > 35) | (ages >= 5) & (resp_rate > 24), resp_rate > 20)
    spo2_flag = spo2 < 92.0
    has_high_risk_vitals = (hr_flag | rr_flag | spo2_flag).astype(int)

    # 6. ESI-v5 Simulated Acuity (Automatic upgrade of 3, 4, 5 if high-risk vitals present)
    esi_v5 = esi_v4.copy()
    uptriaged_mask = (esi_v4 >= 3) & (has_high_risk_vitals == 1)
    esi_v5[uptriaged_mask] = 2

    # 7. Clinical Outcomes & Complications (Ground Truth Targets)
    # ICU risk influenced strongly by acuity, frailty, and high-risk vitals
    icu_prob = np.where(esi_v4 == 1, 0.65,
               np.where(esi_v4 == 2, 0.12,
               np.where(esi_v4 == 3, 0.03 + (0.02 * (cfs_scores >= 5)), 0.005)))
    admitted_to_icu = (np.random.rand(n_samples) < icu_prob).astype(int)

    # 30-day mortality risk
    mort_prob = np.where(esi_v4 == 1, 0.225,
                np.where(esi_v4 == 2, 0.031 + (0.03 * (cfs_scores >= 5)),
                np.where(esi_v4 == 3, 0.016 + (0.02 * (cfs_scores >= 5)), 0.001)))
    mortality_30d = (np.random.rand(n_samples) < mort_prob).astype(int)

    # Resource utilization (count of labs, imaging, IVs, consults)
    resources_used = np.where(esi_v4 == 1, np.random.choice([4, 5], size=n_samples, p=[0.2, 0.8]),
                     np.where(esi_v4 == 2, np.random.choice([2, 3, 4], size=n_samples, p=[0.25, 0.50, 0.25]),
                     np.where(esi_v4 == 3, np.random.choice([1, 2, 3], size=n_samples, p=[0.20, 0.50, 0.30]),
                     np.where(esi_v4 == 4, 1, 0))))

    # 8. Queue Dynamics & Dynamic Scheduler Features
    # Baseline arrival wait time (minutes) and surge factor simulation (3x volume)
    base_wait_time_mins = np.random.exponential(scale=np.where(esi_v4 <= 2, 10, 45), size=n_samples)
    is_surge_shift = np.random.choice([0, 1], size=n_samples, p=[0.75, 0.25]) # 25% of shifts are surges
    current_wait_time_mins = np.where(is_surge_shift == 1, base_wait_time_mins * 3.0, base_wait_time_mins)

    # Clinician Overrides (Nurse overrides model when ambiguity or subtle frailty presents)
    # Target: captures at least 5-10% realistic overrides
    override_occurred = np.where((cfs_scores >= 6) & (esi_v5 >= 3), 1,
                        np.where((has_prior_history == 0) & (has_high_risk_vitals == 1) & (esi_v5 >= 3), 1, 0))

    df = pd.DataFrame({
        'patient_id': patient_ids,
        'age': ages,
        'age_cohort': age_cohort,
        'gender': genders,
        'cfs_frailty_score': cfs_scores,
        'has_prior_history': has_prior_history,
        'comorbidity_count': comorbidity_counts,
        'heart_rate': np.round(heart_rate, 1),
        'resp_rate': np.round(resp_rate, 1),
        'spo2': np.round(spo2, 1),
        'sbp': np.round(sbp, 1),
        'temp_c': np.round(temp_c, 1),
        'has_high_risk_vitals': has_high_risk_vitals,
        'esi_v4_triage': esi_v4,
        'esi_v5_triage': esi_v5,
        'resources_used': resources_used,
        'current_wait_time_mins': np.round(current_wait_time_mins, 1),
        'is_surge_shift': is_surge_shift,
        'override_occurred': override_occurred,
        'admitted_to_icu': admitted_to_icu,
        'mortality_30d': mortality_30d
    })

    return df

df_synthetic = generate_synthetic_triage_data()

In [138]:
df_synthetic

,patient_id,age,age_cohort,gender,cfs_frailty_score,has_prior_history,comorbidity_count,heart_rate,resp_rate,spo2,...,temp_c,has_high_risk_vitals,esi_v4_triage,esi_v5_triage,resources_used,current_wait_time_mins,is_surge_shift,override_occurred,admitted_to_icu,mortality_30d
0,PID_00001,37,adult,Male,2,1,0,78.5,21.2,98.6,...,37.7,1,3,2,1,134.5,1,0,0,0
1,PID_00002,90,geriatric,Female,3,0,0,71.2,17.1,99.4,...,37.3,0,5,5,0,22.0,1,0,0,0
2,PID_00003,85,geriatric,Male,2,0,0,105.1,21.2,94.0,...,38.8,1,2,2,4,17.4,0,0,0,0
3,PID_00004,32,adult,Female,2,0,0,74.0,18.0,99.0,...,35.8,0,3,3,3,1.7,0,0,0,0
4,PID_00005,48,adult,Male,1,1,0,79.2,16.1,95.8,...,37.1,0,3,3,1,157.6,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6225,PID_06226,48,adult,Female,1,1,2,85.4,13.6,97.8,...,37.9,0,3,3,3,19.6,0,0,0,0
6226,PID_06227,10,pediatric,Female,1,1,0,92.4,18.9,97.7,...,36.6,0,4,4,1,23.2,0,0,0,0
6227,PID_06228,73,geriatric,Female,2,1,1,80.9,18.4,96.9,...,37.6,0,3,3,1,4.6,0,0,0,0
6228,PID_06229,94,geriatric,Male,5,1,3,65.5,17.1,97.5,...,36.8,0,4,4,1,938.9,1,0,0,0


GERIATRIC FINAL MODEL

In [139]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

# 1. Isolate the Geriatric Cohort and Define the Objective Target
df_geriatric = df_synthetic[df_synthetic['age_cohort'] == 'geriatric'].copy()
df_geriatric['critical_outcome'] = ((df_geriatric['admitted_to_icu'] == 1) |
                                    (df_geriatric['mortality_30d'] == 1)).astype(int)

# 2. Custom Asymmetric Objective Function (Penalty multiplier alpha = 23.0)
def asymmetric_logistic_obj(preds, dmatrix):
    labels = dmatrix.get_label()
    p = 1.0 / (1.0 + np.exp(-preds))

    alpha = 23.0
    beta = 1.0

    grad = p * (alpha * labels + beta * (1 - labels)) - alpha * labels
    hess = p * (1 - p) * (alpha * labels + beta * (1 - labels))
    return grad, hess

# 3. Prepare the Feature Space and Train/Test Split
features = [
    'heart_rate', 'resp_rate', 'spo2', 'sbp', 'temp_c',
    'cfs_frailty_score', 'has_prior_history', 'comorbidity_count'
]
X = df_geriatric[features]
y = df_geriatric['critical_outcome']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

# 4. Train the Geriatric Boosting Agent
params = {
    'max_depth': 4,
    'learning_rate': 0.01,
    'tree_method': 'hist',
    'min_child_weight': 1,
    'disable_default_eval_metric': 1
}

geriatric_agent = xgb.train(
    params,
    dtrain,
    num_boost_round=150,
    obj=asymmetric_logistic_obj
)

# 5. Apply the Cost-Sensitive Optimal Threshold (0.504)
cost_sensitive_threshold = 0.504

raw_margins = geriatric_agent.predict(dtest)
probabilities = 1.0 / (1.0 + np.exp(-raw_margins))
y_pred_class = (probabilities >= cost_sensitive_threshold).astype(int)

# 6. Evaluation and Clinical Metric Reporting
y_true = y_test.values
tn, fp, fn, tp = confusion_matrix(y_true, y_pred_class).ravel()

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
undertriage_rate = fn / (tp + fn) if (tp + fn) > 0 else 0
overtriage_rate = fp / (tn + fp) if (tn + fp) > 0 else 0
roc_auc = roc_auc_score(y_true, probabilities)

print("--- COST-OPTIMIZED GERIATRIC AGENT REPORT ---")
print(f"Applied Threshold: {cost_sensitive_threshold}")
print(f"ROC-AUC Score: {roc_auc:.3f}")
print(f"True Positives (Critical Caught): {tp}")
print(f"True Negatives (Stable Standardized): {tn}")
print(f"False Negatives (Undertriage / Safety Risk): {fn} ({undertriage_rate*100:.1f}%)")
print(f"False Positives (Overtriage / Operational Friction): {fp} ({overtriage_rate*100:.1f}%)\n")
print(classification_report(y_true, y_pred_class, target_names=['Standard (ESI 3+)', 'Critical (ESI 1/2)'], zero_division=0))

# 7. Generate Frontend UI Payload with Confidence and Human Override Triggers
def generate_ui_payload(probabilities, threshold):
    results = []
    for p in probabilities:
        confidence = (1.0 - 2.0 * abs(p - 0.5)) * 100
        ai_recommendation = 'ESCALATE (ESI 2)' if p >= threshold else 'STANDARD (ESI 3+)'
        requires_override = True if confidence < 20.0 else False

        results.append({
            'risk_probability': round(float(p), 3),
            'confidence_score': round(float(confidence), 1),
            'ai_recommendation': ai_recommendation,
            'requires_human_review': requires_override
        })
    return results

frontend_payload = generate_ui_payload(probabilities, cost_sensitive_threshold)

--- COST-OPTIMIZED GERIATRIC AGENT REPORT ---
Applied Threshold: 0.504
ROC-AUC Score: 0.728
True Positives (Critical Caught): 28
True Negatives (Stable Standardized): 165
False Negatives (Undertriage / Safety Risk): 4 (12.5%)
False Positives (Overtriage / Operational Friction): 169 (50.6%)

                    precision    recall  f1-score   support

 Standard (ESI 3+)       0.98      0.49      0.66       334
Critical (ESI 1/2)       0.14      0.88      0.24        32

          accuracy                           0.53       366
         macro avg       0.56      0.68      0.45       366
      weighted avg       0.90      0.53      0.62       366



ADULT FINAL MODEL

In [140]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, precision_recall_curve

# 1. Isolate the Adult Cohort and Define Objective Critical Outcomes (ICU Admission / 30-Day Mortality)
df_adult = df_synthetic[df_synthetic['age_cohort'] == 'adult'].copy()
df_adult['critical_outcome'] = ((df_adult['admitted_to_icu'] == 1) |
                                (df_adult['mortality_30d'] == 1)).astype(int)

# 2. Custom Asymmetric Objective Function
# Adult-calibrated penalty multiplier: alpha = 18.0 (Undertriage safety bias), beta = 1.0 (Overtriage weight)
def adult_asymmetric_logistic_obj(preds, dmatrix):
    labels = dmatrix.get_label()
    p = 1.0 / (1.0 + np.exp(-preds))

    alpha = 18.0
    beta = 1.0

    grad = p * (alpha * labels + beta * (1 - labels)) - alpha * labels
    hess = p * (1 - p) * (alpha * labels + beta * (1 - labels))
    return grad, hess

# 3. Feature Selection & Train/Test Matrix Preparation
# Omits geriatric-specific CFS score; focuses on acute physiological derangement and history
features = [
    'heart_rate', 'resp_rate', 'spo2', 'sbp', 'temp_c',
    'has_prior_history', 'comorbidity_count'
]
X = df_adult[features]
y = df_adult['critical_outcome']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

# 4. Train the Adult Boosting Agent
params = {
    'max_depth': 4,
    'learning_rate': 0.01,
    'tree_method': 'hist',
    'min_child_weight': 1,
    'disable_default_eval_metric': 1
}

adult_agent = xgb.train(
    params,
    dtrain,
    num_boost_round=150,
    obj=adult_asymmetric_logistic_obj
)

# 5. Extract Probabilities & Optimal Operational Decision Boundary
raw_margins = adult_agent.predict(dtest)
probabilities = 1.0 / (1.0 + np.exp(-raw_margins))
y_true = y_test.values

# Automated Cost-Sensitive Threshold Selection (Penalizing FN 20x over FP)
precisions, recalls, thresholds = precision_recall_curve(y_true, probabilities)
cost_fn, cost_fp = 20.0, 1.0
best_cost = float('inf')
optimal_threshold = 0.50

for th in thresholds:
    y_temp = (probabilities >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_temp).ravel()
    current_cost = (cost_fn * fn) + (cost_fp * fp)
    if current_cost < best_cost:
        best_cost = current_cost
        optimal_threshold = th

# 6. Evaluation & Clinical Metric Reporting
y_pred_class = (probabilities >= optimal_threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_true, y_pred_class).ravel()

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
undertriage_rate = fn / (tp + fn) if (tp + fn) > 0 else 0
overtriage_rate = fp / (tn + fp) if (tn + fp) > 0 else 0
roc_auc = roc_auc_score(y_true, probabilities)

print("--- ADULT DEMOGRAPHIC AGENT EVALUATION REPORT ---")
print(f"Optimal Operating Threshold: {optimal_threshold:.3f}")
print(f"ROC-AUC Score: {roc_auc:.3f}\n")
print(f"True Positives (Critical Escalated): {tp}")
print(f"True Negatives (Stable Standardized): {tn}")
print(f"False Negatives (Undertriage / Safety Risk): {fn} ({undertriage_rate * 100:.1f}%)")
print(f"False Positives (Overtriage / Operational Friction): {fp} ({overtriage_rate * 100:.1f}%)\n")
print("--- ASYMMETRIC METRICS ---")
print(f"Sensitivity (Catch Rate): {sensitivity * 100:.1f}%")
print(f"Specificity: {specificity * 100:.1f}%\n")
print(classification_report(y_true, y_pred_class, target_names=['Standard (ESI 3+)', 'Critical (ESI 1/2)'], zero_division=0))

# 7. Generate Unified Frontend UI Payload
def generate_adult_payload(probs, threshold):
    results = []
    for p in probs:
        confidence = (1.0 - 2.0 * abs(p - 0.5)) * 100
        ai_recommendation = 'ESCALATE (ESI 2)' if p >= threshold else 'STANDARD (ESI 3+)'
        requires_override = True if confidence < 20.0 else False

        results.append({
            'risk_probability': round(float(p), 3),
            'confidence_score': round(float(confidence), 1),
            'ai_recommendation': ai_recommendation,
            'requires_human_review': requires_override
        })
    return results

patient_scores_adult = generate_adult_payload(probabilities, optimal_threshold)

--- ADULT DEMOGRAPHIC AGENT EVALUATION REPORT ---
Optimal Operating Threshold: 0.543
ROC-AUC Score: 0.805

True Positives (Critical Escalated): 52
True Negatives (Stable Standardized): 385
False Negatives (Undertriage / Safety Risk): 8 (13.3%)
False Positives (Overtriage / Operational Friction): 242 (38.6%)

--- ASYMMETRIC METRICS ---
Sensitivity (Catch Rate): 86.7%
Specificity: 61.4%

                    precision    recall  f1-score   support

 Standard (ESI 3+)       0.98      0.61      0.75       627
Critical (ESI 1/2)       0.18      0.87      0.29        60

          accuracy                           0.64       687
         macro avg       0.58      0.74      0.52       687
      weighted avg       0.91      0.64      0.71       687



PEDIATRIC FINAL MODEL

In [141]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

# 1. Isolate Pediatric Cohort and Define Objective Target
df_pediatric = df_synthetic[df_synthetic['age_cohort'] == 'pediatric'].copy()
df_pediatric['critical_outcome'] = ((df_pediatric['admitted_to_icu'] == 1) |
                                    (df_pediatric['mortality_30d'] == 1)).astype(int)

# 2. Custom Asymmetric Objective Function (Pediatric Alpha = 25.0)
def pediatric_asymmetric_logistic_obj(preds, dmatrix):
    labels = dmatrix.get_label()
    p = 1.0 / (1.0 + np.exp(-preds))

    alpha = 28.0
    beta = 1.0

    grad = p * (alpha * labels + beta * (1 - labels)) - alpha * labels
    hess = p * (1 - p) * (alpha * labels + beta * (1 - labels))
    return grad, hess

# 3. Prepare Feature Space and Train/Test Split (Including age for pediatric baselines)
features = [
    'age', 'heart_rate', 'resp_rate', 'spo2', 'sbp', 'temp_c',
    'has_prior_history', 'comorbidity_count'
]
X = df_pediatric[features]
y = df_pediatric['critical_outcome']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

# 4. Standard Non-Regularized Hyperparameters for Pediatrics
params = {
    'max_depth': 4,
    'learning_rate': 0.01,
    'tree_method': 'hist',
    'min_child_weight': 1,
    'disable_default_eval_metric': 1
}

pediatric_agent = xgb.train(
    params,
    dtrain,
    num_boost_round=150,
    obj=pediatric_asymmetric_logistic_obj
)

# 5. Apply Baseline Threshold
threshold = 0.504

raw_margins = pediatric_agent.predict(dtest)
probabilities = 1.0 / (1.0 + np.exp(-raw_margins))
y_pred_class = (probabilities >= threshold).astype(int)

# 6. Evaluation and Clinical Metric Reporting
y_true = y_test.values
tn, fp, fn, tp = confusion_matrix(y_true, y_pred_class).ravel()

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
undertriage_rate = fn / (tp + fn) if (tp + fn) > 0 else 0
overtriage_rate = fp / (tn + fp) if (tn + fp) > 0 else 0
roc_auc = roc_auc_score(y_true, probabilities)

print("--- NON-REGULATED PEDIATRIC AGENT REPORT ---")
print(f"Applied Threshold: {threshold}")
print(f"ROC-AUC Score: {roc_auc:.3f}")
print(f"True Positives (Critical Caught): {tp}")
print(f"True Negatives (Stable Standardized): {tn}")
print(f"False Negatives (Undertriage / Safety Risk): {fn} ({undertriage_rate*100:.1f}%)")
print(f"False Positives (Overtriage / Operational Friction): {fp} ({overtriage_rate*100:.1f}%)\n")
print(classification_report(y_true, y_pred_class, target_names=['Standard (ESI 3+)', 'Critical (ESI 1/2)'], zero_division=0))

--- NON-REGULATED PEDIATRIC AGENT REPORT ---
Applied Threshold: 0.504
ROC-AUC Score: 0.695
True Positives (Critical Caught): 12
True Negatives (Stable Standardized): 93
False Negatives (Undertriage / Safety Risk): 4 (25.0%)
False Positives (Overtriage / Operational Friction): 85 (47.8%)

                    precision    recall  f1-score   support

 Standard (ESI 3+)       0.96      0.52      0.68       178
Critical (ESI 1/2)       0.12      0.75      0.21        16

          accuracy                           0.54       194
         macro avg       0.54      0.64      0.44       194
      weighted avg       0.89      0.54      0.64       194



In [148]:
import numpy as np
import xgboost as xgb

def evaluate_esi_v5_abcde_floor(patient_data):
    """
    Deterministic ABCDE safety floor rule engine implementing ESI Version 5
    criteria. Acts as an uncompromisable rule-based safety net decoupled from ML probabilities.
    """
    min_esi_floor = 5  # Default lowest acuity
    triggered_rules = []

    # Handle both pandas Series/DataFrame row or dict input
    if hasattr(patient_data, 'to_dict'):
        p = patient_data.to_dict()
    else:
        p = patient_data

    age = p.get('age', 35)
    cohort = p.get('age_cohort', 'adult')
    is_pediatric = (cohort == 'pediatric') or (age < 18)
    is_geriatric = (cohort == 'geriatric') or (age >= 65)

    # Decision Point A: Immediate life-saving interventions required (ESI Level 1)
    if p.get('requires_lifesaving_intervention', 0) == 1:
        return {
            'acuity_floor': 1,
            'triggered_rules': ["Decision Point A: Immediate life-saving intervention required"],
            'is_hard_locked': True
        }

    # Decision Point B: High-risk situation, altered mental status, or severe distress (ESI Level 2)
    if p.get('high_risk_situation', 0) == 1 or \
       p.get('altered_mental_status', 0) == 1 or \
       p.get('severe_pain_distress', 0) == 1:
        min_esi_floor = min(min_esi_floor, 2)
        triggered_rules.append("Decision Point B: High-risk presentation / Altered mental status / Severe distress")

    # Decision Point D: High-Risk Vital Signs Assessment
    hr = p.get('heart_rate')
    rr = p.get('resp_rate')
    spo2 = p.get('spo2')

    if hr is not None and rr is not None and spo2 is not None:
        if is_pediatric:
            hr_limit = 140 if age < 5 else 110
            rr_limit = 35 if age < 5 else 24
            if hr > hr_limit or rr > rr_limit or spo2 < 92.0:
                min_esi_floor = min(min_esi_floor, 2)
                triggered_rules.append(f"Decision Point D (Pediatric): Critical vital derangement (HR:{hr}, RR:{rr}, SpO2:{spo2})")
        else:
            # Adult & Geriatric ESI-v5 limits: HR > 100, RR > 20, SpO2 < 92%
            if hr > 100 or rr > 20 or spo2 < 92.0:
                min_esi_floor = min(min_esi_floor, 2)
                triggered_rules.append(f"Decision Point D (Adult/Geriatric): High-risk vital signs (HR:{hr}, RR:{rr}, SpO2:{spo2})")

    # Geriatric Frailty Guard (Clinical Frailty Scale >= 5 shifts physiological reserve)
    cfs = p.get('cfs_frailty_score', 1)
    if is_geriatric and cfs >= 5:
        if hr is not None and hr > 90:
            min_esi_floor = min(min_esi_floor, 2)
            triggered_rules.append(f"Geriatric Vulnerability Guard: Elevated heart rate ({hr} bpm) with CFS Frailty Score {cfs}")

    return {
        'acuity_floor': min_esi_floor,
        'triggered_rules': triggered_rules,
        'is_hard_locked': min_esi_floor <= 2
    }

In [150]:
def compute_confidence_tree_variance(model, patient_features, num_boost_rounds=150, step=15):
    """
    Method 2: Tree-Level Ensemble Variance Confidence Score.
    Evaluates prediction consensus across sequential boosting stages (sub-ranges of trees).
    High variance across iterations indicates conflicting tree signals and model uncertainty.
    """
    # Format features into a DMatrix
    d_input = xgb.DMatrix(np.array([patient_features]))

    stage_predictions = []
    # Sample prediction evolution across boosting iterations
    for k in range(step, num_boost_rounds + 1, step):
        raw_margin = model.predict(d_input, iteration_range=(0, k))[0]
        prob = 1.0 / (1.0 + np.exp(-raw_margin))
        stage_predictions.append(prob)

    # Calculate standard deviation across tree stage predictions
    std_dev = np.std(stage_predictions)

    # Map standard deviation to a 0-100 confidence score (higher std_dev = lower confidence)
    # Max theoretical std for probabilities is ~0.5; normalize accordingly
    normalized_uncertainty = min(1.0, std_dev / 0.25)
    confidence_score = (1.0 - normalized_uncertainty) * 100.0

    return max(0.0, min(100.0, confidence_score)), stage_predictions